# Chakaso — Kaggle CPU benchmark

**What this measures.** How fast this repository's *real* training loop runs a gradient update of the
[E2636](../../experiments/E2636/record.md) architecture on a Kaggle free-tier CPU notebook, plus held-out
evaluation time, peak resident memory, checkpoint write/read cost, and the fingerprints of the tokenizer and
corpus it actually used.

Every number comes from the code the measured records are produced by: `chakaso.training.nn_training.train_transformer`,
`chakaso.training.decoder.ChakasoDecoder`, `chakaso.training.packing.pack_units`,
`chakaso.training.nn_evaluation.evaluate_model`, `chakaso.training.nn_checkpoint.save_artifact`. There is no
re-implementation of the model, the optimiser, the batch plan or the metrics anywhere in this notebook, and no
stand-in PyTorch workload.

**What this is not.** This is **not E2636, and it does not run any of its six arms.** E2636 is a preregistered
experiment whose record states it has not run; nothing here touches its protocol files, its seeds, its bands, its
decision rules or its status. The benchmark runs a handful of steps at the same architecture and the same
declared optimiser recipe, on a small slice of text, and names every setting it reduced — see *Declared versus
reduced*.

**Why it exists.** E2636's cost paragraph is arithmetic on rates measured on one Windows host at
`torch_threads: 14` (5,834–6,759 positions/s, 10.1–11.7 h for six arms). Whether a Kaggle CPU notebook could hold
that session is an unmeasured question, and the answer is a multiplier on those rates. This notebook produces the
multiplier in about two minutes, so a ~12-hour commitment can be decided from a measurement instead of a hope.

**Read this before trusting the number.** A rate measured on 4 threads is not comparable to one measured on 14 as
a property of the model — `describe_environment` records the thread count precisely because
`nn_checkpoint`'s own docstring calls the reduction order part of the numerical procedure. What this notebook
supports is *Kaggle-CPU versus this-machine-Windows*, both read off the same code path, and it is valid only if
the environment block it prints is kept with the result.

## Running it on Kaggle

1. **Upload the repository as a Kaggle Dataset.** This needs `src/chakaso/`,
   `configs/experiments/E2636-adopted-budget-7200.toml`, `experiments/E2620/vocabulary-4a47f0281c81a9e5.json`
   and `experiments/E2612/vocabulary-434debe82f949347.json`. The discovery cell looks for
   `src/chakaso/training/nn_training.py` rather than assuming a mount path.
2. **Accelerator: None.** There is no GPU code path in `chakaso.training`: nothing calls `.cuda()`, no
   `torch.device("cuda")` is constructed, `load_artifact` forces `map_location="cpu"`, and
   `describe_environment()` writes `device: cpu` as a literal. A GPU selection would change nothing here and
   would make the figure not comparable with the CPU records. The Dependency check cell asserts CUDA is absent.
3. **Attach the corpus only if you want the real fingerprint.** Set `CORPUS_SNAPSHOT` below, or leave it `None`
   and the cell searches `/kaggle/input` for `gutenberg-multiregister-v1-snapshot.json` (157 MB — the free tier
   mounts it read-only, it is never copied). Without it the benchmark trains on this repository's own tracked
   markdown, labelled as such, and the corpus fingerprint it verifies is that working-tree manifest, **not**
   E2636's `312571f9fa8351a8`. Step throughput is unaffected, because a step's shape is fixed by
   `batch_size × context_length`; the held-out bits-per-character is not comparable across corpora.
4. **Persistence off is fine.** Two minutes of compute.
5. Kaggle ships `torch`. If the dependency cell reports `numpy: absent`, install it in a scratch cell —
   `!{sys.executable} -m pip install --quiet numpy` — and nothing else.

**Outputs** go to `/kaggle/working` on Kaggle and `data/benchmarks/kaggle-cpu/` locally (git-ignored). Never into
`experiments/`, which holds committed records; a guard cell asserts that rather than trusting the reader.

In [ ]:
from pathlib import Path

# --- the only cell an operator edits -------------------------------------------------
# Sizes are stated in the units the E2636 protocol uses, so the reduction is visible.
WARMUP_STEPS = 3  # gradient updates timed and discarded
MEASURED_STEPS = 12  # gradient updates that produce the reported rate
EVAL_BLOCKS = 128  # packed held-out examples scored once, for evaluation timing
TRAIN_TOKEN_CAP = 160_000  # identifiers packed for training (E2636 packs 14,033,424)
EVAL_TOKEN_CAP = 40_000  # identifiers packed for held-out timing
CORPUS_SNAPSHOT: Path | None = None  # or a path to gutenberg-multiregister-v1-snapshot.json
OUT_DIR: Path | None = None  # None -> /kaggle/working on Kaggle, data/benchmarks/kaggle-cpu here

In [ ]:
import importlib.metadata
import json
import os
import platform
import shutil
import socket
import statistics
import sys
import time

import torch

KAGGLE = Path("/kaggle/working").is_dir()
SNAPSHOT_NAME = "gutenberg-multiregister-v1-snapshot.json"


def _find_repo_root() -> Path:
    """Find the checkout by marker, not by a mount path.

    On Kaggle an uploaded dataset lands at /kaggle/input/<dataset>/ and the notebook's cwd is /kaggle/working;
    locally the notebook sits inside the repository. The marker is the training module's own file, because a
    directory containing it is the project and a directory with a similar name is not.
    """
    marker = Path("src/chakaso/training/nn_training.py")
    mounts = sorted(p for p in Path("/kaggle/input").glob("*") if p.is_dir()) if KAGGLE else []
    for seed in [Path.cwd(), *mounts, *(d for m in mounts for d in sorted(m.glob("*"))[:24])]:
        for candidate in (seed, seed / "chakaso", seed / "chakaso-main"):
            if (candidate / marker).is_file():
                return candidate.resolve()
    message = (
        "no Chakaso checkout found: looked for src/chakaso/training/nn_training.py under "
        f"{Path.cwd()} and under /kaggle/input. Upload the repository as a dataset."
    )
    raise FileNotFoundError(message)


REPO = _find_repo_root()
# `import chakaso` must resolve to THIS checkout, not to whatever wheel the image happens to carry: a
# benchmark measured against an installed copy of different code would be a number about nothing.
sys.path.insert(0, str(REPO / "src"))

print(f"repository root : {REPO}")
print(f"kaggle          : {KAGGLE}")
print(f"host            : {socket.gethostname()} on {platform.platform()}")
print(f"python          : {sys.version.split()[0]} at {sys.executable}")

In [ ]:
import chakaso
from chakaso.training.nn_checkpoint import describe_environment

environment = describe_environment()
for name, value in sorted(environment.items()):
    print(f"{name:<15}: {value}")

for distribution in ("chakaso", "numpy", "torch"):
    try:
        version = importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        version = "ABSENT -- pip install it before continuing"
    print(f"dist {distribution:<12}: {version}")

print(f"package file    : {Path(chakaso.__file__).resolve()}")
assert environment["device"] == "cpu"
assert not torch.cuda.is_available(), (
    "a CUDA device is visible; this probe must run on CPU to compare"
)
assert Path(chakaso.__file__).resolve().is_relative_to(REPO), (
    "the imported chakaso is not the checkout under test -- the sys.path insert above did not take"
)

In [ ]:
from chakaso.core.process_memory import available_memory, peak_memory

machine = available_memory()
before = peak_memory()
disk_target = Path("/kaggle/working") if KAGGLE else REPO
disk = shutil.disk_usage(disk_target)
print(f"logical cpus (os)  : {os.cpu_count()}")
print(f"torch threads      : {torch.get_num_threads()}")
print(f"torch interop      : {torch.get_num_interop_threads()}")
free = machine.available_bytes
print(
    f"physical memory    : {(machine.total_bytes or 0) / 2**30:.2f} GiB total, "
    f"{(free or 0) / 2**30:.2f} GiB free  [{machine.source}]"
)
print(f"disk at output     : {disk.free / 2**30:.1f} GiB free of {disk.total / 2**30:.1f} GiB")
print(
    f"process peak (now) : {(before.peak_resident_bytes or 0) / 2**20:.1f} MiB  [{before.source}]"
)
print(f"cpu model          : {platform.processor() or platform.machine()}")

In [ ]:
OUT_DIR = OUT_DIR or (Path("/kaggle/working") if KAGGLE else REPO / "data/benchmarks/kaggle-cpu")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR = OUT_DIR.resolve()

# E2636's constraint, enforced by code rather than remembered: a benchmark must not land beside a committed
# record, where a later reader could mistake it for one.
assert (REPO / "experiments").resolve() not in OUT_DIR.parents, (
    f"refusing to write benchmark output inside the committed records at {OUT_DIR}"
)
print(f"output directory  : {OUT_DIR}")

## Fingerprints, read from the committed artifacts

Both vocabularies E2636 declares are **committed in the repository**, so they ship with the upload and their
identities are verified here against the values in the protocol file — not assumed from a filename.
`SubwordVocabulary.load` re-checks the file's own checksum before returning, so a truncated or edited merge
table fails here rather than becoming a benchmark result.

In [ ]:
from chakaso.training.subword import SubwordVocabulary

TOKENIZER_FILE = REPO / "experiments/E2620/vocabulary-4a47f0281c81a9e5.json"
SELECTION_TOKENIZER_FILE = REPO / "experiments/E2612/vocabulary-434debe82f949347.json"

vocabulary = SubwordVocabulary.load(TOKENIZER_FILE)
selection_vocabulary = SubwordVocabulary.load(SELECTION_TOKENIZER_FILE)

print(
    f"training vocabulary     : {vocabulary.fingerprint}  size={vocabulary.size} "
    f"min_frequency={vocabulary.min_frequency}"
)
print(f"  pre_tokenizer         : {vocabulary.pre_tokenizer}")
print(f"  file                  : {TOKENIZER_FILE.relative_to(REPO)}")
print(
    f"evaluation-selection    : {selection_vocabulary.fingerprint}  "
    f"size={selection_vocabulary.size}"
)
print(f"  pre_tokenizer         : {selection_vocabulary.pre_tokenizer}")

assert vocabulary.fingerprint == "4a47f0281c81a9e5", (
    f"E2636's pinned training vocabulary is 4a47f0281c81a9e5, loaded {vocabulary.fingerprint}"
)
assert selection_vocabulary.fingerprint == "434debe82f949347", (
    f"E2636's pinned selection vocabulary is 434debe82f949347, loaded "
    f"{selection_vocabulary.fingerprint}"
)

## Corpus, and what its fingerprint proves

`read_snapshot` re-hashes every document in the snapshot against the manifest it carries, so a fingerprint that
equals E2636's is a statement about 149,000,499 characters of bytes, not about a filename. If no snapshot is
attached the cell falls back to this repository's tracked markdown, through the same `build_manifest` the pinned
`chakaso-repo-text-v1` corpus was made with, and says so in the result file.

In [ ]:
from chakaso.training.manifest import build_manifest, read_snapshot
from chakaso.training.subword_experiment import REPO_TEXT_CORPUS

E2636_CORPUS_FINGERPRINT = "312571f9fa8351a8"


def _find_snapshot() -> Path | None:
    if CORPUS_SNAPSHOT is not None:
        return Path(CORPUS_SNAPSHOT)
    hits = sorted(Path("/kaggle/input").rglob(SNAPSHOT_NAME)) if KAGGLE else []
    hits += sorted(REPO.rglob(SNAPSHOT_NAME))
    return hits[0] if hits else None


snapshot_path = _find_snapshot()
if snapshot_path is not None:
    snapshot = read_snapshot(snapshot_path)
    manifest = snapshot.manifest
    corpus_origin = "the pinned E2636 snapshot, re-hashed document by document"
    fingerprint_matches = manifest.fingerprint == E2636_CORPUS_FINGERPRINT
    if not fingerprint_matches:
        print(
            f"WARNING: snapshot at {snapshot_path} hashes to {manifest.fingerprint}, "
            f"not E2636's {E2636_CORPUS_FINGERPRINT} -- this is a different corpus"
        )
else:
    manifest = build_manifest(
        REPO,
        name=REPO_TEXT_CORPUS.name,
        version=REPO_TEXT_CORPUS.version,
        licence=REPO_TEXT_CORPUS.licence,
        source=REPO_TEXT_CORPUS.source,
        include=REPO_TEXT_CORPUS.include,
        extensions=REPO_TEXT_CORPUS.extensions,
    )
    snapshot, corpus_origin = None, "this repository's tracked markdown (NOT E2636's corpus)"
    fingerprint_matches = False

units = manifest.text_units()
print(f"corpus source           : {corpus_origin}")
print(f"corpus name/version     : {manifest.name} {manifest.version}")
print(f"fingerprint             : {manifest.fingerprint}")
print(f"== E2636's declared     : {fingerprint_matches}")
print(f"documents               : {len(manifest.documents)}  characters: {manifest.char_count:,}")
print(f"split-ready units       : {len(units)}")

In [ ]:
from chakaso.training.data import split_text_units
from chakaso.training.packing import pack_units, select_units_by_token_cap


def _capped_sample(units, reference, cap, *, minimum_works=3, stride=1):
    """Whole units from across the corpus, under ``cap``, spanning at least ``minimum_works`` works.

    E2636 splits by whole work at split seed 7, and `split_text_units` refuses a whole-work split over fewer
    than three works. Capping in manifest order would take the corpus's first 37 chunks -- two books, because
    a book's chunks are adjacent -- so the benchmark strides the corpus before capping. The units are fewer,
    they come from across the whole corpus, and the rule that assigns sides is the record's own.
    """
    while True:
        chosen = select_units_by_token_cap(units[::stride], reference, max_tokens=cap)
        works = {unit.group for unit in chosen}
        if len(works) >= minimum_works or stride >= len(units):
            return chosen, stride, works
        stride *= 4


sampled, stride, works = _capped_sample(units, vocabulary, TRAIN_TOKEN_CAP + EVAL_TOKEN_CAP)
# E2636's split rule, not a convenience one: whole-work separation at split seed 7.
split = split_text_units(
    sampled, validation_fraction=0.15, test_fraction=0.15, seed=7, whole_works=True
)
train_pack = pack_units(split.train, vocabulary, context_length=256)
held_out = pack_units(split.test, vocabulary, context_length=256)
print(f"stride over the corpus  : every {stride}th unit")
print(f"units sampled           : {len(sampled)} from {len(units)}, across {len(works)} works")
print(
    f"train / validation / test units: {len(split.train)} / {len(split.validation)} / {len(split.test)}"
)
print(f"dropped duplicates      : {len(split.dropped_duplicates)}")
print(
    f"train blocks            : {len(train_pack.examples)} of 256 positions "
    f"({train_pack.tokens_available:,} identifiers)"
)
print(f"held-out blocks         : {len(held_out.examples)}")
print(f"split fingerprint       : {split.fingerprint}")
assert train_pack.examples and held_out.examples, "nothing packed: the corpus selects no text"
leaked = {u.identifier for u in split.train} & {u.identifier for u in split.test}
assert not leaked, f"leakage: the same document is on both sides: {sorted(leaked)[:3]}"

## Declared versus reduced

The configuration is **read from E2636's own protocol file** through `chakaso.training.protocol.load_plan`, the
loader the experiment uses, and passed through `architecture_for` and `_training_config` — the same two functions
`run_experiment` calls. Nothing below is retyped; the table prints what the record declares against what this
benchmark runs, so a reduction is visible instead of smuggled.

In [ ]:
from dataclasses import replace

from chakaso.training.protocol import load_plan
from chakaso.training.subword_experiment import _training_config, architecture_for

PROTOCOL = REPO / "configs/experiments/E2636-adopted-budget-7200.toml"
plan = load_plan(
    PROTOCOL,
    root=REPO,
    work_dir=OUT_DIR,
    fixed={
        "split_seed": 7,
        "expected_corpus_fingerprint": E2636_CORPUS_FINGERPRINT,
        "corpus_snapshot": snapshot_path,
    },
)
architecture = architecture_for(plan, vocabulary)
declared = _training_config(plan, architecture)

# The benchmark's own ceiling: warm-up then measurement, one evaluation per step so the loop's own
# per-interval rate is reported at step granularity. positions_per_second is measured over gradient work
# ONLY (nn_training's definition), so evaluating every step does not inflate it.
steps = WARMUP_STEPS + MEASURED_STEPS
reduced = replace(
    declared,
    max_steps=steps,
    eval_every=1,
    warmup_steps=min(declared.warmup_steps, max(1, steps // 2)),
    checkpoint_every=steps,
)

REDUCED_FIELDS = ("max_steps", "eval_every", "warmup_steps", "checkpoint_every")
print(f"{'field':<24}{'E2636 declares':>18}{'this benchmark':>18}")
for name in REDUCED_FIELDS:
    print(f"{name:<24}{getattr(declared, name)!s:>18}{getattr(reduced, name)!s:>18}")
UNCHANGED = (
    "architecture",
    "batch_size",
    "grad_accum_steps",
    "learning_rate",
    "weight_decay",
    "schedule",
    "decay_to",
    "gradient_clip_norm",
    "patience",
    "min_delta",
    "document_masking",
    "seed",
)
print(f"\n{'unchanged from the record ->':<24}")
for name in UNCHANGED:
    print(f"{name:<24}{getattr(reduced, name)!s:>18}")

assert reduced.architecture == declared.architecture == architecture
assert architecture.parameter_count == 3_413_760, (
    f"E2636's reference configuration is 3,413,760 parameters, got {architecture.parameter_count:,}"
)
assert reduced.batch_size == 32 and reduced.max_steps == steps
print(f"\nparameter count       : {architecture.parameter_count:,}")
print(f"positions per step    : {reduced.batch_size * architecture.context_length:,}")
print(
    f"projected gradient work: {steps * reduced.batch_size * architecture.context_length:,} positions"
)

In [ ]:
from chakaso.training.nn_training import train_transformer

# Warm-up inside the same run: the first updates pay lazy dispatch, thread-pool spin-up and the first
# matmul planning. Those steps are recorded and then excluded from the reported rate, which is why the
# loop is asked for a rate at every step.
started = time.perf_counter()
outcome = train_transformer(
    train_examples=train_pack.examples,
    validation_examples=held_out.examples,
    vocabulary=vocabulary,
    config=reduced,
    corpus_fingerprint=manifest.fingerprint,
    experiment="chakaso-cpubench",
    split_fingerprint=split.fingerprint,
)
run_seconds = time.perf_counter() - started

per_step = [record for record in outcome.records if record.step > 0]
warm = [r.positions_per_second for r in per_step[:WARMUP_STEPS]]
measured = [r.positions_per_second for r in per_step[WARMUP_STEPS:]]
print(
    f"wall clock (whole probe) : {run_seconds:.2f} s over {steps} updates "
    f"(includes {steps} evaluations)"
)
print(f"warm-up rates (excluded) : {[round(v, 1) for v in warm]}")
print(f"measured rates           : {[round(v, 1) for v in measured]}")
print(f"loop's own rate (all steps, gradient time only): {outcome.positions_per_second:,.1f} pos/s")

In [ ]:
from chakaso.training.nn_checkpoint import checkpoint_filename, load_artifact, save_artifact

artifact = outcome.artifact()
checkpoint = OUT_DIR / checkpoint_filename("chakaso-cpubench", plan.model_family)
model = artifact.to_model()

wrote = time.perf_counter()
save_artifact(artifact, checkpoint)
write_seconds = time.perf_counter() - wrote
read = time.perf_counter()
reloaded = load_artifact(checkpoint)
read_seconds = time.perf_counter() - read

print(f"artifact               : {checkpoint.name}  {checkpoint.stat().st_size / 2**20:.1f} MiB")
print(f"save_artifact          : {write_seconds:.3f} s")
print(f"load_artifact          : {read_seconds:.3f} s")
print(f"checksum agrees        : {reloaded.checksum() == artifact.checksum()}")
assert reloaded.checksum() == artifact.checksum(), "a round-trip changed the artifact"
assert reloaded.vocabulary.fingerprint == vocabulary.fingerprint
artifact_mib = round(checkpoint.stat().st_size / 2**20, 2)
# The probe writes no optimizer state, so this is a usable artifact: 17 MiB, not the 43 MiB a
# resumable one costs. It is not an experiment record and must not linger in the output directory.
checkpoint.unlink()

In [ ]:
from chakaso.training.nn_evaluation import evaluate_model

eval_examples = held_out.examples[:EVAL_BLOCKS]
forward = time.perf_counter()
with torch.no_grad():
    metrics = evaluate_model(model, eval_examples, vocabulary, batch_size=declared.batch_size)
eval_seconds = time.perf_counter() - forward
positions = metrics.positions_scored
print(f"held-out blocks scored : {len(eval_examples)}")
print(f"positions              : {positions:,}  characters: {metrics.characters_scored:,}")
print(
    f"evaluate_model         : {eval_seconds:.2f} s  ({positions / eval_seconds:,.0f} pos/s forward)"
)
print(
    f"loss                   : {metrics.loss_nats:.4f} nats/token  "
    f"perplexity {metrics.perplexity:.2f}"
)
print(
    f"bits per character     : {metrics.bits_per_character:.4f}  <- NOT comparable across corpora"
)
print(f"token accuracy         : {metrics.accuracy:.4f}")

In [ ]:
after = peak_memory()
print(
    f"process peak resident  : {(after.peak_resident_bytes or 0) / 2**20:.1f} MiB  [{after.source}]"
)
print(f"process now resident   : {(after.resident_bytes or 0) / 2**20:.1f} MiB")

rate = statistics.median(measured)
mean_rate = sum(measured) / len(measured)
long_arm_positions = 7200 * declared.batch_size * architecture.context_length
short_arm_positions = 2400 * declared.batch_size * architecture.context_length
result = {
    "schema": "chakaso-kaggle-cpu-benchmark/1",
    "is_not_e2636": "this probe trains a handful of steps; no E2636 arm was launched by it",
    "environment": environment,
    "machine": {
        "hostname": socket.gethostname(),
        "platform": platform.platform(),
        "cpu_model": platform.processor() or platform.machine(),
        "logical_cpus": os.cpu_count(),
        "torch_threads": torch.get_num_threads(),
        "physical_total_bytes": machine.total_bytes,
        "physical_available_bytes": machine.available_bytes,
        "memory_source": machine.source,
        "disk_free_bytes": disk.free,
        "disk_total_bytes": disk.total,
    },
    "process_memory_bytes": {
        "peak_before": before.peak_resident_bytes,
        "peak_after": after.peak_resident_bytes,
        "resident_after": after.resident_bytes,
        "source": after.source,
    },
    "fingerprints": {
        "training_tokenizer": vocabulary.fingerprint,
        "training_tokenizer_expected": "4a47f0281c81a9e5",
        "training_tokenizer_matches": vocabulary.fingerprint == "4a47f0281c81a9e5",
        "selection_tokenizer": selection_vocabulary.fingerprint,
        "selection_tokenizer_expected": "434debe82f949347",
        "selection_tokenizer_matches": selection_vocabulary.fingerprint == "434debe82f949347",
        "corpus": manifest.fingerprint,
        "corpus_e2636_expected": E2636_CORPUS_FINGERPRINT,
        "corpus_matches_e2636": fingerprint_matches,
        "corpus_origin": corpus_origin,
        "split": split.fingerprint,
    },
    "settings": {
        "protocol": str(PROTOCOL.relative_to(REPO)),
        # `architecture` is the DecoderConfig itself; it is recorded through its own serialisation, which is
        # the same mapping the checkpoint's metadata document carries.
        "declared": {
            name: (architecture.as_dict() if name == "architecture" else getattr(declared, name))
            for name in (*REDUCED_FIELDS, *UNCHANGED)
        },
        "reduced_for_benchmark": {name: getattr(reduced, name) for name in REDUCED_FIELDS},
        "architecture": architecture.as_dict(),
        "parameter_count": architecture.parameter_count,
        "positions_per_step": reduced.batch_size * architecture.context_length,
        "warmup_steps_excluded": WARMUP_STEPS,
        "measured_steps": MEASURED_STEPS,
        "train_blocks": len(train_pack.examples),
        "eval_blocks_scored": len(eval_examples),
    },
    "measured": {
        "median_positions_per_second": round(rate, 1),
        "mean_positions_per_second": round(mean_rate, 1),
        "slowest_step_positions_per_second": round(min(measured), 1),
        "fastest_step_positions_per_second": round(max(measured), 1),
        "per_step_positions_per_second": [round(v, 1) for v in measured],
        "warmup_positions_per_second": [round(v, 1) for v in warm],
        "loop_reported_positions_per_second": round(outcome.positions_per_second, 1),
        "run_wall_clock_seconds": round(run_seconds, 3),
        "evaluation_seconds": round(eval_seconds, 3),
        "evaluation_positions_per_second": round(positions / eval_seconds, 1),
        "eval_positions": positions,
        "held_out_loss_nats": round(metrics.loss_nats, 6),
        "held_out_bits_per_character": round(metrics.bits_per_character, 6),
        "token_accuracy": round(metrics.accuracy, 6),
        "artifact_mib": artifact_mib,
        "usable_artifact_write_seconds": round(write_seconds, 3),
        "usable_artifact_read_seconds": round(read_seconds, 3),
    },
    "projection_for_e2636": {
        "note": (
            "arithmetic on the rates measured above, at E2636's own declared update counts, batch shape and "
            "evaluation cap. It is a projection for THIS host, not a result. Each arm evaluates 19 times over "
            "its own ceiling at eval_every 400, scoring the declared 250,000-identifier pack each time; the "
            "forward rate measured here prices that, because this probe's held-out pack is smaller than the "
            "record's and its seconds are not transferable -- its positions per second are."
        ),
        "short_arm_seconds_at_2400": round(short_arm_positions / rate, 1) if rate else None,
        "long_arm_seconds_at_7200": round(long_arm_positions / rate, 1) if rate else None,
        "evaluations_per_arm": {"short": 7, "long": 19},
        "one_evaluation_seconds_at_250000_positions": (
            round(250_000 / (positions / eval_seconds), 1) if eval_seconds else None
        ),
        "six_arms_gradient_only_hours": (
            round(3 * (short_arm_positions + long_arm_positions) / rate / 3600, 2) if rate else None
        ),
        "six_arms_including_evaluation_hours": (
            round(
                3
                * (
                    short_arm_positions / rate
                    + 7 * 250_000 / (positions / eval_seconds)
                    + long_arm_positions / rate
                    + 19 * 250_000 / (positions / eval_seconds)
                )
                / 3600,
                2,
            )
            if rate and eval_seconds
            else None
        ),
    },
}

result_path = OUT_DIR / "kaggle-cpu-benchmark.json"
result_path.write_text(json.dumps(result, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print(f"\nwrote {result_path}")

In [ ]:
summary = OUT_DIR / "kaggle-cpu-benchmark.md"
p = result["projection_for_e2636"]
m = result["measured"]
f = result["fingerprints"]
mem = result["process_memory_bytes"]
mac = result["machine"]

# Locals rather than nested conditionals inside f-strings: the sentence an operator reads is checked here,
# once, and the table below stays readable.
tokenizer_verdict = (
    "matches the declared 4a47f0281c81a9e5"
    if f["training_tokenizer_matches"]
    else ("DOES NOT match the declared 4a47f0281c81a9e5")
)
selection_verdict = "matches" if f["selection_tokenizer_matches"] else "DOES NOT match"
corpus_verdict = (
    "IS the pinned E2636 corpus" if f["corpus_matches_e2636"] else "IS NOT the pinned E2636 corpus"
)
lines = [
    "# Chakaso CPU benchmark — result",
    "",
    f"Host `{mac['hostname']}` on `{mac['platform']}`, {mac['logical_cpus']} logical CPUs, "
    f"torch on {result['environment']['torch_threads']} threads, "
    f"torch {result['environment']['torch']}, python {result['environment']['python']}.",
    "",
    "| Measurement | Value |",
    "| --- | --- |",
    f"| gradient throughput, median of measured steps | **{m['median_positions_per_second']:,.0f} pos/s** |",
    f"| ... mean / spread | {m['mean_positions_per_second']:,.0f} pos/s, "
    f"{m['slowest_step_positions_per_second']:,.0f}-{m['fastest_step_positions_per_second']:,.0f} |",
    f"| loop-reported throughput, all steps | {m['loop_reported_positions_per_second']:,.0f} pos/s |",
    f"| held-out evaluation, forward only | {m['evaluation_positions_per_second']:,.0f} pos/s "
    f"({m['evaluation_seconds']:.2f} s over {m['eval_positions']:,} positions) |",
    f"| peak resident | {(mem['peak_after'] or 0) / 2**20:,.0f} MiB [{mem['source']}] |",
    f"| free RAM at launch | {(mac['physical_available_bytes'] or 0) / 2**30:.2f} GiB |",
    f"| usable artifact write / read | {m['usable_artifact_write_seconds']:.3f} s / "
    f"{m['usable_artifact_read_seconds']:.3f} s ({m['artifact_mib']} MiB) |",
    "",
    f"**Fingerprints.** Training tokenizer `{f['training_tokenizer']}` — {tokenizer_verdict}. "
    f"Selection tokenizer `{f['selection_tokenizer']}` — {selection_verdict}. "
    f"Corpus `{f['corpus']}` — {corpus_verdict} ({f['corpus_origin']}). "
    f"Split `{f['split']}`.",
    "",
    f"**Projection for E2636 at this host's rates** (arithmetic, not a run): a 2,400-step arm "
    f"{round(p['short_arm_seconds_at_2400'] / 3600, 2)} h, a 7,200-step arm "
    f"{round(p['long_arm_seconds_at_7200'] / 3600, 2)} h; all six arms "
    f"{p['six_arms_gradient_only_hours']} h on gradient work alone, "
    f"{p['six_arms_including_evaluation_hours']} h including the record's own evaluation load "
    f"(one 250,000-position evaluation costs {p['one_evaluation_seconds_at_250000_positions']} s here). "
    "The Windows reference in the record is 10.1-11.7 h.",
    "",
    "**No E2636 arm was launched by this notebook.** It trains a handful of steps at the record's declared "
    "architecture and optimiser recipe, reducing only `max_steps`, `eval_every`, `warmup_steps` and "
    "`checkpoint_every`, all printed in the notebook. The held-out bits per character is scored on this "
    "probe's sampled pack, not the record's capped 250,000-identifier pack, and is not a comparison with "
    "any measured record.",
    "",
    f"_Schema `{result['schema']}`; machine-readable copy beside this file._",
]
summary.write_text("\n".join(lines) + "\n", encoding="utf-8")
print(f"wrote {summary}")
print()
print("\n".join(lines))

## Reading the result

- **`median of measured steps`** is what the projection is built on, and the spread beside it is why. Two runs
  of this same notebook on the same machine, minutes apart, reported `loop_reported_positions_per_second` of
  **6,502 and 2,579** — a 2.5× swing — because the first steps of a process pay lazy dispatch, thread-pool
  spin-up and first-touch of ~1.8 GB of pages. `positions_per_second` is measured over gradient work only
  (`nn_training`'s definition), so excluding the warm-up steps is legitimate; averaging eight of them was not
  stable. Twelve steps and a median, with min and max printed, is what a session decision can be made on.
- **The projection assumes the median rate holds across a run.** The measured Windows arms spanned
  5,834–6,759 positions/s across seeds, a 16 % band, and this probe's own step-to-step band is wider than
  that. Treat a projected session length as a range, and read the first pair's measured rate before committing
  the rest — which is the gate [E2636's record](../../experiments/E2636/record.md) already declares.
- **`peak resident`** is what the free-tier question turns on. The Windows arms peaked at 2,501.6–2,540.7 MiB
  on a 16 GiB host. A ten-step probe reaches ~1.85 GiB, and the full run's high-water mark is set by the model,
  both Adam moment sets and the activation block, not by the step count — so read this as a floor, not the peak.
- **A Kaggle figure is comparable to the record's only if `torch_threads` matches.** It will not: Kaggle's free
  CPU instance is 4 vCPUs against this host's 14. That is the finding, not an error to correct — it is why a
  cross-host number has to travel with its environment block, and why
  [K-040](../agent/KNOWN_ISSUES.md) matters for anything that resumes a checkpoint between machines.